# 🧠 Tactile SDF Reconstruction — Google Colab

This notebook trains a **PointNet + SIREN** model to reconstruct 3D shapes from sparse tactile contact points.

**Architecture:**
- **Encoder:** PointNet (processes 6 contact points with position, normal, tangent).
- **Decoder:** SIREN (implicit neural representation mapping `(x, y, z) + latent` to SDF).

**Dataset:** `grasp-dataset-gen` (Objaverse objects).

In [ ]:
# @title 🛠️ Step 1: Setup & Dependencies
!pip install trimesh rtree plotly streamlit scikit-image scikit-learn tqdm
!pip install fast-simplification  # For faster mesh decimation

In [ ]:
# @title 📂 Step 2: Clone Repositories
import os

# Clone the tactile-sdf repo
if not os.path.exists('tactile-sdf'):
    !git clone https://github.com/635jack/tactile-sdf.git

# Clone the dataset repo (or assume it's uploaded)
if not os.path.exists('grasp-dataset-gen'):
    !git clone https://github.com/635jack/grasp-dataset-gen.git

%cd tactile-sdf

> [!NOTE]
> **Data Note**: If you have your data on Google Drive, mount it and symlink the directories:
> ```python
> from google.colab import drive
> drive.mount('/content/drive')
> !ln -s /content/drive/MyDrive/grasp-dataset-gen ../grasp-dataset-gen
> ```

In [ ]:
# @title 📐 Step 3: Run SDF Preprocessing
# This computes ground truth SDF from GLB meshes
!python preprocess_sdf.py --n_points 50000 --glb_dir ../grasp-dataset-gen/data/objaverse --dataset_dir ../grasp-dataset-gen/output_hf

In [ ]:
# @title 🚀 Step 4: Start Training
# Recommended: 300-500 epochs for high quality
!python train.py --epochs 300 --batch_size 8 --n_query 2048 --eval_every 10 --dataset_dir ../grasp-dataset-gen/output_hf

In [ ]:
# @title 📊 Step 5: Visualize Results
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

# Find latest run
runs = sorted(glob.glob('runs/*'))
if runs:
    latest_run = runs[-1]
    img_path = os.path.join(latest_run, 'training_curves.png')
    if os.path.exists(img_path):
        plt.figure(figsize=(15, 10))
        img = mpimg.imread(img_path)
        plt.imshow(img)
        plt.axis('off')
        plt.show()
else:
    print("No training runs found.")